In [ ]:
#test_vis_params
#Grok prompt:
#geemap python landsat iterate over a selection of vis_params and save results as png


# --------------------------------------------------------------
# 1. Install / import
# --------------------------------------------------------------
# !pip install -q geemap  # run once in Colab / Jupyter

import ee
import geemap
import os
from pathlib import Path

# --------------------------------------------------------------
# 2. Authenticate & initialize Earth Engine
# --------------------------------------------------------------
ee.Authenticate()   # only needed the first time
ee.Initialize()

In [ ]:
# --------------------------------------------------------------
# 3. Define the area of interest (AOI)
# --------------------------------------------------------------
# Example: a polygon around Anchorage, AK (you can replace with any geometry)
aoi = ee.Geometry.Polygon(
    [[[-149.5, 61.0],
      [-149.5, 61.5],
      [-148.5, 61.5],
      [-148.5, 61.0],
      [-149.5, 61.0]]])

# --------------------------------------------------------------
# 4. Build the Landsat image collection
# --------------------------------------------------------------
landsat = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')   # Landsat 8
           .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))  # Landsat 9
           .filterBounds(aoi)
           .filterDate('2023-06-01', '2023-09-01')   # summer season
           .sort('CLOUD_COVER')
           .first())                         # pick the least cloudy image

# Apply scaling factors (SR data are in 1/10000)
def apply_scale_factors(image):
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(opticalBands, None, True) \
                .addBands(thermalBands, None, True)

landsat = apply_scale_factors(landsat)

# --------------------------------------------------------------
# 5. List of visualization parameters to iterate over
# --------------------------------------------------------------
vis_params_list = [
    {
        "name": "true_color",
        "params": {"bands": ['SR_B4', 'SR_B3', 'SR_B2'], "min": 0, "max": 0.3}
    },
    {
        "name": "false_color_nir",
        "params": {"bands": ['SR_B5', 'SR_B4', 'SR_B3'], "min": 0, "max": 0.3}
    },
    {
        "name": "agriculture",
        "params": {"bands": ['SR_B6', 'SR_B5', 'SR_B2'], "min": 0, "max": 0.3}
    },
    {
        "name": "thermal",
        "params": {"bands": ['ST_B10'], "min": 290, "max": 320, "palette": ['blue', 'white', 'red']}
    }
]

# --------------------------------------------------------------
# 6. Output folder
# --------------------------------------------------------------
out_dir = Path("landsat_vis_exports")
out_dir.mkdir(exist_ok=True)

# --------------------------------------------------------------
# 7. Iterate, render, and export PNGs
# --------------------------------------------------------------
for item in vis_params_list:
    name   = item["name"]
    vis    = item["params"]
    
    # Create a temporary Map object (so each export is independent)
    Map = geemap.Map(center=[61.25, -149.0], zoom=9)
    
    # Add the Landsat layer with the current vis params
    Map.addLayer(landsat.clip(aoi), vis, name)
    
    # OPTIONAL: add a basemap for context
    Map.add_basemap('ROADMAP')
    
    # Define the export file path
    png_path = out_dir / f"{name}.png"
    
    # Export the map view as PNG
    #   region = aoi (clip to the AOI)
    #   dimensions = 1200x1200 (adjust as you like)
    #   scale = 30 m (Landsat native resolution)
    Map.to_png(
        filename=str(png_path),
        region=aoi,
        dimensions=(1200, 1200),
        scale=30,
        crs='EPSG:4326'
    )
    
    print(f"Exported: {png_path}")

print("\nAll PNGs saved to:", out_dir.resolve())